# Fine-Tuning de YOLOv8 — Billetes Mexicanos (TT2026-2_IA16)

Notebook reproducible de entrenamiento y evaluacion. Entrenamiento con descongelamiento gradual en 3 fases, semilla fija. Pensado para Kaggle / Google Colab con GPU (T4 / P100).

**Estado:** base. La ejecucion de las corridas y la evaluacion se completan con el dataset etiquetado.

## 1. Instalacion de dependencias

In [ ]:
!pip install ultralytics roboflow -q

## 2. Imports y semilla fija (reproducibilidad)

In [ ]:
import os, random
import numpy as np
import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print('GPU disponible:', torch.cuda.is_available())
print('Dispositivo:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 3. Dataset

Descarga del dataset exportado de Roboflow (formato YOLOv8). Sustituye los datos por los de tu workspace/proyecto/version. El archivo `data.yaml` define las 6 clases y las rutas train/val/test.

In [ ]:
# from roboflow import Roboflow
# rf = Roboflow(api_key='TU_API_KEY')
# project = rf.workspace('TU_WORKSPACE').project('TU_PROYECTO')
# dataset = project.version(1).download('yolov8')
# DATA_YAML = f'{dataset.location}/data.yaml'

DATA_YAML = 'data.yaml'  # ajusta a la ruta del dataset descargado
print('Config del dataset:', DATA_YAML)

In [ ]:
from pathlib import Path
from collections import Counter
import yaml

root = Path(dataset.location)
names = yaml.safe_load(open(root/'data.yaml'))['names']

ann, imgs_clase = Counter(), Counter()
total, vacias, no_bbox = 0, 0, 0

for split in ['train','valid','test']:
    d = root/split/'labels'
    if not d.exists(): continue
    for f in d.glob('*.txt'):
        total += 1
        lineas = [l.split() for l in f.read_text().splitlines() if l.strip()]
        if not lineas:
            vacias += 1; continue
        for l in lineas:
            if len(l) != 5: no_bbox += 1        # 5 valores = bounding box
        clases = [int(l[0]) for l in lineas]
        ann.update(clases)
        imgs_clase.update(set(clases))

print(f"Imágenes: {total} | sin etiqueta: {vacias} | etiquetas NO bbox: {no_bbox}")
print("  (no bbox debe ser 0: confirma que los contornos se convirtieron a cajas)\n")
print(f"{'Clase':<16}{'Anotaciones':>12}{'Imágenes':>11}")
for i, n in enumerate(names):
    print(f"{n:<16}{ann[i]:>12}{imgs_clase[i]:>11}")
print(f"{'TOTAL':<16}{sum(ann.values()):>12}")

for s in ['train','valid','test']:
    p = root/s/'images'
    print(f"{s}: {len(list(p.glob('*'))) if p.exists() else 0}")

## 4. Entrenamiento con descongelamiento gradual (3 fases)

| Fase | Capas entrenables | Epocas | LR |
|------|-------------------|--------|----|
| 1 | Solo Head (deteccion) | 10 | 1e-3 |
| 2 | Head + Neck | 15 | 1e-4 |
| 3 | Red completa | 25 | 1e-5 |

Config comun: YOLOv8s pre-entrenado en COCO, entrada 640, batch 16, optimizador AdamW, weight decay 0.01, early stopping (patience 10). Misma configuracion que el reporte tecnico.

In [ ]:
COMMON = dict(data=DATA_YAML, imgsz=640, batch=16, optimizer='AdamW',
              weight_decay=0.01, patience=10, seed=SEED, device=0, exist_ok=True)

# --- Fase 1: solo la cabeza de deteccion (backbone congelado) ---
model = YOLO('yolov8s.pt')
model.train(**COMMON, epochs=10, lr0=1e-3, freeze=10,
            project='billetes', name='fase1_head')

In [ ]:
# --- Fase 2: Head + Neck (backbone aun congelado) ---
model = YOLO('billetes/fase1_head/weights/best.pt')
model.train(**COMMON, epochs=15, lr0=1e-4, freeze=10,
            project='billetes', name='fase2_neck')

In [ ]:
# --- Fase 3: red completa (sin congelar) ---
model = YOLO('billetes/fase2_neck/weights/best.pt')
model.train(**COMMON, epochs=25, lr0=1e-5, freeze=0,
            project='billetes', name='fase3_full')

## 5. Evaluacion (mAP, precision, recall y matriz de confusion)

Se evalua sobre el conjunto de **prueba** (no aumentado). Objetivos: mAP@0.5 >= 85%, precision y recall >= 80%.

In [ ]:
best = YOLO('billetes/fase3_full/weights/best.pt')
metrics = best.val(data=DATA_YAML, split='test', plots=True)
print('mAP@0.5    :', metrics.box.map50)
print('mAP@0.5:.95:', metrics.box.map)
print('Precision  :', metrics.box.mp)
print('Recall     :', metrics.box.mr)
# La matriz de confusion se guarda como imagen en la carpeta de resultados (confusion_matrix.png).

## 6. Conteo de billetes y monto total

A partir de las detecciones se cuenta cada denominacion y se suma el monto total.

In [ ]:
VALOR = {'billete_20':20,'billete_50':50,'billete_100':100,
         'billete_200':200,'billete_500':500,'billete_1000':1000}

def contar(imagen):
    r = best(imagen)[0]
    nombres = r.names
    conteo = {}
    for c in r.boxes.cls.tolist():
        clase = nombres[int(c)]
        conteo[clase] = conteo.get(clase, 0) + 1
    total = sum(VALOR[k]*v for k, v in conteo.items())
    return conteo, total

# conteo, total = contar('ejemplo.jpg')
# print('Detecciones:', conteo, '| Monto total: $', total)